# Task 01 — Production-Ready RAG Chatbot

This notebook explores the complete pipeline for building a production-ready
RAG-based chatbot using LangChain and related tools.

We will move step-by-step from document ingestion to chatbot tracing,
understanding both the theory and practical implementation of each component.

## **1. Data Ingestion** 

In [ ]:
from pathlib import Path 

pdf_path = Path("data/pdf/attention_is_all_you_need.pdf") 


WindowsPath('data/pdf/attention_is_all_you_need.pdf')

In [3]:
# PyPDFLoader 
from langchain_community.document_loaders import PyPDFLoader 
pdf_loader = PyPDFLoader(pdf_path)
pdf_docs = pdf_loader.load() 

print(f"Number of documents loaded: {len(pdf_docs)}") 
print(f"Metadata of first Document object: \n{pdf_docs[0].metadata}") 
print(f"\nPage content: {pdf_docs[0].page_content}")

Number of documents loaded: 11
Metadata of first Document object: 
{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the e

In [6]:
# We h've diagram on page 3 let's see it what happened with diagram when we use PyPDFLoader 
print(pdf_docs[3].page_content)

Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
query with all keys, divide each by√dk, and apply a softmax function to obtain the weights on the
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
into a matrixQ. The keys and values are also packed together into matricesK andV . We compute
the matrix of outputs as:
Attention(Q,K,V ) = softmax(QK T
√dk
)V (1)
The two most commonly used attention functions are additive attention [2], and dot-product (multi-
plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor
of 1√dk
. Additive attention computes the compatibility function using a feed-forward network with
a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is
much faster and more space-efﬁcient in pra

In [15]:
print(pdf_docs[8].page_content)

Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to
per-word perplexities.
N d model dff h d k dv Pdrop ϵls
train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36
4 5.19 25.3 50
8 4.88 25.5 80
256 32 32 5.75 24.5 28
1024 128 128 4.66 26.0 168
1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility

********* 


NOTE 
* Extract text only from PDF files.(not helpful when our pdf have images, diagram or complex layout like tables etc)
* It typically processes the PDF page by page and creates a Document object for each page.
* Each Document contains the extracted text (page_content) and PDF-related metadata, such as the source and page number 
* Extract content from the table but not preserve the original shape of the table 

In [9]:
from langchain_community.document_loaders import PyPDFium2Loader

loader = PyPDFium2Loader(pdf_path) 
docs = loader.load() 

print(f"Number of Document: {len(docs)}")
print(f"\nMetadata: {docs[0].metadata}")
print(f"\nPage content of first doc:\n{docs[0].page_content}")

Number of Document: 11

Metadata: {'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'source': 'data\\pdf\\attention_is_all_you_need.pdf', 'total_pages': 11, 'page': 0}

Page content of first doc:
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent 

In [11]:
print(docs[3].page_content)

Scaled Dot-Product Attention Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
query with all keys, divide each by √
dk, and apply a softmax function to obtain the weights on the
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
into a matrix Q. The keys and values are also packed together into matrices K and V . We compute
the matrix of outputs as:
Attention(Q, K, V ) = softmax(QKT
√
dk
)V (1)
The two most commonly used attention functions are additive attention [2], and dot-product (multi￾plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor
of √
1
dk
. Additive attention computes the compatibility function using a feed-forward network with
a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is
much faster and more space-efficien

In [ ]:
print(docs[8].page_content) # We've table on this page let's see how it handles the tables 

Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to
per-word perplexities.
N dmodel dff h dk dv Pdrop ls
train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36
4 5.19 25.3 50
8 4.88 25.5 80
256 32 32 5.75 24.5 28
1024 128 128 4.66 26.0 168
1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility i

* **PyPDFium2Loader** is helpful when you need to load PDFs using the **PDFium engine** through `pypdfium2`.
* It is useful for PDFs where **page rendering and visual layout** are important, such as PDFs containing images, graphics, or complex page layouts.
* It can be a good alternative to `PyPDFLoader` when **PDFium-based processing** provides better results for a particular PDF.


In [16]:
from langchain_community.document_loaders import PyMuPDFLoader 

loader = PyMuPDFLoader(pdf_path) 
docs = loader.load() 

print(len(docs))

11


In [17]:
docs[0].metadata

{'producer': 'PyPDF2',
 'creator': '',
 'creationdate': '',
 'source': 'data\\pdf\\attention_is_all_you_need.pdf',
 'file_path': 'data\\pdf\\attention_is_all_you_need.pdf',
 'total_pages': 11,
 'format': 'PDF 1.3',
 'title': 'Attention is All you Need',
 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
 'subject': 'Neural Information Processing Systems http://nips.cc/',
 'keywords': '',
 'moddate': '2018-02-12T21:22:10-08:00',
 'trapped': '',
 'modDate': "D:20180212212210-08'00'",
 'creationDate': '',
 'page': 0}

In [20]:
print(docs[0].page_content[:1000])

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model a

In [22]:
print(docs[8].page_content)

Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to
per-word perplexities.
N
dmodel
dff
h
dk
dv
Pdrop
ϵls
train
PPL
BLEU
params
steps
(dev)
(dev)
×106
base
6
512
2048
8
64
64
0.1
0.1
100K
4.92
25.8
65
(A)
1
512
512
5.29
24.9
4
128
128
5.00
25.5
16
32
32
4.91
25.8
32
16
16
5.01
25.4
(B)
16
5.16
25.1
58
32
5.01
25.4
60
(C)
2
6.11
23.7
36
4
5.19
25.3
50
8
4.88
25.5
80
256
32
32
5.75
24.5
28
1024
128
128
4.66
26.0
168
1024
5.12
25.4
53
4096
4.75
26.2
90
(D)
0.0
5.77
24.6
0.2
4.95
25.5
0.0
4.67
25.3
0.2
5.47
25.7
(E)
positional embedding instead of sinusoids
4.92
25.7
big
6
1024
4096
16
0.3
300K
4.33
26.4
213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility i

# 2. Text Loader 

In [23]:
from langchain_community.document_loaders import TextLoader 

loader = TextLoader(r"data\txt\vision_transformer.txt")
docs = loader.load() 

len(docs)

1

# 3. Docx Loader 


In [26]:
from langchain_community.document_loaders import Docx2txtLoader 

loader = Docx2txtLoader(r"data\docs\llms_are_few_shot_solver.docx")

docs = loader.load() 
len(docs)

1

# 4. CSV Loader 

In [27]:
from langchain_community.document_loaders import CSVLoader 

loader = CSVLoader(r"data\csv\store.csv")
docs = loader.load() 
len(docs)

20

# 5. Markdown Loader 

In [1]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader 

loader = UnstructuredMarkdownLoader(r"data\markdown\sample.markdown") 
docs = loader.load() 
len(docs)

1

# **Module 2. Splitters**

I'll explore RecursiveCharacterTextSplitter only here

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.document_loaders import PyPDFLoader 

# 1. load data 
loader = PyPDFLoader(r"data\pdf\2010.11929v2.pdf")
docs = loader.load()

len(docs)

22

In [31]:
# 2. splitter  
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, 
    chunk_overlap=100
) 

chunks = splitter.split_documents(docs) 


In [38]:
# Total number of chunks
print("Total chunks:", len(chunks))

# Inspect first 5 chunks
for i, chunk in enumerate(chunks[:5]):
    print(f"\n{'='*60}")
    print(f"CHUNK {i+1}")
    print(f"Characters: {len(chunk.page_content)}")
    print(f"Content:\n{chunk.page_content}")

Total chunks: 104

CHUNK 1
Characters: 784
Content:
Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16 X16 W ORDS :
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗,
Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer,
Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,†
∗equal technical contribution,†equal advising
Google Research, Brain Team
{adosovitskiy, neilhoulsby}@google.com
ABSTRACT
While the Transformer architecture has become the de-facto standard for natural
language processing tasks, its applications to computer vision remain limited. In
vision, attention is either applied in conjunction with convolutional networks, or
used to replace certain components of convolutional networks while keeping their

CHUNK 2
Characters: 726
Content:
used to replace certain components of convolutional networks while keeping their
overall structure in place. We show that this re

In [39]:
chunk_sizes = [len(chunk.page_content) for chunk in chunks]

print("Total chunks:", len(chunk_sizes))
print("Maximum chunk size:", max(chunk_sizes))
print("Minimum chunk size:", min(chunk_sizes))
print("Average chunk size:", sum(chunk_sizes) / len(chunk_sizes))

# Check chunks larger than configured chunk_size
large_chunks = [
    (i + 1, size)
    for i, size in enumerate(chunk_sizes)
    if size > 800
]

print("\nChunks larger than 800 characters:", len(large_chunks))

if large_chunks:
    print(large_chunks[:10])

Total chunks: 104
Maximum chunk size: 800
Minimum chunk size: 65
Average chunk size: 698.2980769230769

Chunks larger than 800 characters: 0


In [41]:
for i in range(len(chunks) - 1):
    current = chunks[i].page_content
    next_chunk = chunks[i + 1].page_content

    # Find the longest suffix of current chunk
    # that appears as a prefix of the next chunk
    max_overlap = min(len(current), len(next_chunk), 100)

    overlap = ""

    for n in range(max_overlap, 0, -1):
        if current[-n:] == next_chunk[:n]:
            overlap = current[-n:]
            break

    print(f"\n{'='*60}")
    print(f"CHUNK {i+1} → CHUNK {i+2}")
    print(f"Overlap characters: {len(overlap)}")

    if overlap:
        print(f"Overlap text:\n{repr(overlap)}")


CHUNK 1 → CHUNK 2
Overlap characters: 80
Overlap text:
'used to replace certain components of convolutional networks while keeping their'

CHUNK 2 → CHUNK 3
Overlap characters: 98
Overlap text:
'Self-attention-based architectures, in particular Transformers (Vaswani et al., 2017), have become'

CHUNK 3 → CHUNK 4
Overlap characters: 98
Overlap text:
'Krizhevsky et al., 2012; He et al., 2016). Inspired by NLP successes, multiple works try combining'

CHUNK 4 → CHUNK 5
Overlap characters: 99
Overlap text:
'2020).\nInspired by the Transformer scaling successes in NLP, we experiment with applying a standard'

CHUNK 5 → CHUNK 6
Overlap characters: 94
Overlap text:
'seemingly discouraging outcome may be expected: Transformers lack some of the inductive biases'

CHUNK 6 → CHUNK 7
Overlap characters: 0

CHUNK 7 → CHUNK 8
Overlap characters: 89
Overlap text:
'reaches the accuracy of 88.55% on ImageNet, 90.72% on ImageNet-ReaL, 94.55% on CIFAR-100,'

CHUNK 8 → CHUNK 9
Overlap characters: 98
Over

In [42]:
for i, chunk in enumerate(chunks):
    print(
        f"Chunk {i+1:03d} | "
        f"Characters: {len(chunk.page_content):4d} | "
        f"Preview: {chunk.page_content[:80]!r}"
    )

Chunk 001 | Characters:  784 | Preview: 'Published as a conference paper at ICLR 2021\nAN IMAGE IS WORTH 16 X16 W ORDS :\nT'
Chunk 002 | Characters:  726 | Preview: 'used to replace certain components of convolutional networks while keeping their'
Chunk 003 | Characters:  771 | Preview: 'Self-attention-based architectures, in particular Transformers (Vaswani et al., '
Chunk 004 | Characters:  708 | Preview: 'Krizhevsky et al., 2012; He et al., 2016). Inspired by NLP successes, multiple w'
Chunk 005 | Characters:  734 | Preview: '2020).\nInspired by the Transformer scaling successes in NLP, we experiment with '
Chunk 006 | Characters:  249 | Preview: 'seemingly discouraging outcome may be expected: Transformers lack some of the in'
Chunk 007 | Characters:  768 | Preview: 'Published as a conference paper at ICLR 2021\ninherent to CNNs, such as translati'
Chunk 008 | Characters:  711 | Preview: 'reaches the accuracy of 88.55% on ImageNet, 90.72% on ImageNet-ReaL, 94.55% on C'
Chunk 009 | 

# Module 3: Embeddings models 

In [44]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# Create embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Test texts
texts = [
    "Artificial intelligence is transforming software development.",
    "AI is changing the way software is developed.",
    "The weather is very pleasant today."
]

# Generate embeddings
vectors = embeddings.embed_documents(texts)

# Inspect
print("Number of texts:", len(vectors))

for i, vector in enumerate(vectors):
    print(f"\nText {i + 1}: {texts[i]}")
    print("Vector type:", type(vector))
    print("Vector dimension:", len(vector))
    print("First 10 values:", vector[:10])

Number of texts: 3

Text 1: Artificial intelligence is transforming software development.
Vector type: <class 'list'>
Vector dimension: 1536
First 10 values: [-0.0164947509765625, -0.0117645263671875, 0.0141754150390625, -0.01202392578125, 0.0472412109375, -0.03802490234375, -0.0014133453369140625, 0.0517578125, -0.0110931396484375, 0.018341064453125]

Text 2: AI is changing the way software is developed.
Vector type: <class 'list'>
Vector dimension: 1536
First 10 values: [-0.0213623046875, -0.0129241943359375, 0.0278472900390625, 0.0189056396484375, 0.042633056640625, -0.052032470703125, -0.0086212158203125, 0.04229736328125, -0.02081298828125, 0.0161285400390625]

Text 3: The weather is very pleasant today.
Vector type: <class 'list'>
Vector dimension: 1536
First 10 values: [0.0162200927734375, -0.023468017578125, -0.059722900390625, -0.0277862548828125, 0.03240966796875, 0.010162353515625, -0.004047393798828125, 0.00167083740234375, -0.007049560546875, -0.016754150390625]


In [45]:
from numpy import dot
from numpy.linalg import norm


def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))


similarity_1_2 = cosine_similarity(vectors[0], vectors[1])
similarity_1_3 = cosine_similarity(vectors[0], vectors[2])

print("Similarity: Text 1 vs Text 2:", similarity_1_2)
print("Similarity: Text 1 vs Text 3:", similarity_1_3)

Similarity: Text 1 vs Text 2: 0.7906333277704117
Similarity: Text 1 vs Text 3: 0.08047935985834978


In [46]:
query = "How is AI changing software development?"

query_vector = embeddings.embed_query(query)

print("Query vector dimension:", len(query_vector))
print("Document vector dimension:", len(vectors[0]))

Query vector dimension: 1536
Document vector dimension: 1536


### HuggingFace Embeddings 

In [2]:
from sentence_transformers import SentenceTransformer

def get_huggingface_embeddings(model_name: str) -> SentenceTransformer: 

    return SentenceTransformer(model_name) 



texts = [
    "Artificial intelligence is transforming software development.",
    "AI is changing the way software is developed.",
    "The weather is very pleasant today."
]

embd_model = get_huggingface_embeddings("BAAI/bge-small-en-v1.5")

embeddings = embd_model.encode(texts)

print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)
print("Embedding dimension:", embeddings.shape[1])

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

c:\Users\Muhammad Umair\AppData\Local\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Muhammad Umair\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of embeddings: 3
Embedding shape: (3, 384)
Embedding dimension: 384


In [3]:
hf_model2 = get_huggingface_embeddings("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
hf_model3 = get_huggingface_embeddings("intfloat/multilingual-e5-small")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\Muhammad Umair\AppData\Local\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Muhammad Umair\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [5]:
vector = embeddings[0]

print("Vector type:", type(vector))
print("Vector dimension:", len(vector))
print("First 10 values:", vector[:10])

Vector type: <class 'numpy.ndarray'>
Vector dimension: 384
First 10 values: [-0.04827007 -0.02525337  0.04755547 -0.0533863   0.03696157  0.00780349
 -0.02446907  0.06450133  0.00686144 -0.01099179]


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(embeddings)

print(similarities)

[[1.0000004  0.9016347  0.43385532]
 [0.9016347  1.0000001  0.46725762]
 [0.43385532 0.46725762 1.0000002 ]]


In [7]:
print(
    "Text 1 vs Text 2:",
    similarities[0][1]
)

print(
    "Text 1 vs Text 3:",
    similarities[0][2]
)

Text 1 vs Text 2: 0.9016347
Text 1 vs Text 3: 0.43385532


In [8]:
query = "How is artificial intelligence changing software development?"

documents = [
    "AI is transforming modern software engineering.",
    "Pakistan has many beautiful northern areas.",
    "Machine learning is increasingly used in software applications.",
    "The weather is sunny today."
]

query_embedding = embd_model.encode([query])
document_embeddings = embd_model.encode(documents)

scores = cosine_similarity(
    query_embedding,
    document_embeddings
)[0]

for i, score in enumerate(scores):
    print(f"Document {i+1}: {score:.4f}")

Document 1: 0.8550
Document 2: 0.3543
Document 3: 0.7598
Document 4: 0.3295


In [9]:
ranked = sorted(
    zip(documents, scores),
    key=lambda x: x[1],
    reverse=True
)

print("\nRanking:\n")

for rank, (document, score) in enumerate(ranked, start=1):
    print(f"{rank}. {score:.4f} - {document}")


Ranking:

1. 0.8550 - AI is transforming modern software engineering.
2. 0.7598 - Machine learning is increasingly used in software applications.
3. 0.3543 - Pakistan has many beautiful northern areas.
4. 0.3295 - The weather is sunny today.


# **Dense Vs Sparse Vectors**

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "artificial intelligence is transforming software",
    "machine learning is used in software development",
    "the weather is beautiful today"
]

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform(documents)

print("Shape:", vectors.shape)
print("Type:", type(vectors))
print("Vocabulary size:", len(vectorizer.vocabulary_))

print("\nSparse vector:")
print(vectors[0])

Shape: (3, 14)
Type: <class 'scipy.sparse._csr.csr_matrix'>
Vocabulary size: 14

Sparse vector:
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 14)>
  Coords	Values
  (0, 0)	0.5046113401371842
  (0, 4)	0.5046113401371842
  (0, 5)	0.2980315863446099
  (0, 11)	0.5046113401371842
  (0, 8)	0.3837699307603192


In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# -----------------------------
# Documents
# -----------------------------

documents = [
    "The iPhone 15 has a powerful A16 Bionic chip and an advanced camera system.",
    "The Samsung Galaxy S24 has a high resolution display and powerful processor.",
    "Our store provides laptops, smartphones, tablets and other electronic products.",
    "Machine learning models can be deployed using FastAPI and Docker.",
]


query = "Where can I buy an iPhone with a powerful camera?"


# =========================================================
# 1. DENSE SEARCH
# =========================================================

dense_model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode([query])

dense_scores = cosine_similarity(
    query_embedding,
    document_embeddings
)[0]


# =========================================================
# 2. SPARSE SEARCH
# =========================================================

tfidf = TfidfVectorizer()

document_tfidf = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

sparse_scores = cosine_similarity(
    query_tfidf,
    document_tfidf
)[0]


# =========================================================
# 3. NORMALIZE SCORES
# =========================================================

def normalize(scores):
    min_score = min(scores)
    max_score = max(scores)

    return [
        (score - min_score) / (max_score - min_score)
        if max_score != min_score
        else 0
        for score in scores
    ]


dense_scores = normalize(dense_scores)
sparse_scores = normalize(sparse_scores)


# =========================================================
# 4. HYBRID SCORE
# =========================================================

dense_weight = 0.7
sparse_weight = 0.3

hybrid_scores = [
    dense_weight * dense +
    sparse_weight * sparse
    for dense, sparse
    in zip(dense_scores, sparse_scores)
]


# =========================================================
# 5. DISPLAY RESULTS
# =========================================================

results = []

for i, document in enumerate(documents):

    results.append({
        "document": document,
        "dense": dense_scores[i],
        "sparse": sparse_scores[i],
        "hybrid": hybrid_scores[i]
    })


results.sort(
    key=lambda x: x["hybrid"],
    reverse=True
)


for rank, result in enumerate(results, start=1):

    print(f"\nRank {rank}")
    print("-" * 60)

    print("Document:", result["document"])
    print(f"Dense:   {result['dense']:.4f}")
    print(f"Sparse:  {result['sparse']:.4f}")
    print(f"Hybrid:  {result['hybrid']:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Rank 1
------------------------------------------------------------
Document: The iPhone 15 has a powerful A16 Bionic chip and an advanced camera system.
Dense:   1.0000
Sparse:  1.0000
Hybrid:  1.0000

Rank 2
------------------------------------------------------------
Document: The Samsung Galaxy S24 has a high resolution display and powerful processor.
Dense:   0.4445
Sparse:  0.1895
Hybrid:  0.3680

Rank 3
------------------------------------------------------------
Document: Our store provides laptops, smartphones, tablets and other electronic products.
Dense:   0.4433
Sparse:  0.0000
Hybrid:  0.3103

Rank 4
------------------------------------------------------------
Document: Machine learning models can be deployed using FastAPI and Docker.
Dense:   0.0000
Sparse:  0.3026
Hybrid:  0.0908


In [12]:
!pip install rank-bm25 sentence-transformers scikit-learn

In [13]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity


# --------------------------------------------------
# Documents
# --------------------------------------------------

documents = [
    "The iPhone 15 has a powerful A16 Bionic chip and an advanced camera system.",
    "The Samsung Galaxy S24 has a high resolution display and powerful processor.",
    "Our store provides laptops, smartphones, tablets and other electronic products.",
    "Machine learning models can be deployed using FastAPI and Docker.",
]

query = "Where can I buy an iPhone with a powerful camera?"


# ==================================================
# 1. DENSE RETRIEVAL
# ==================================================

dense_model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode([query])

dense_scores = cosine_similarity(
    query_embedding,
    document_embeddings
)[0]


# ==================================================
# 2. BM25 SPARSE RETRIEVAL
# ==================================================

# Tokenize documents
tokenized_documents = [
    document.lower().split()
    for document in documents
]

# Create BM25 index
bm25 = BM25Okapi(tokenized_documents)

# Tokenize query
tokenized_query = query.lower().split()

# Get BM25 scores
bm25_scores = bm25.get_scores(tokenized_query)


# ==================================================
# 3. NORMALIZE SCORES
# ==================================================

def normalize(scores):

    min_score = min(scores)
    max_score = max(scores)

    if max_score == min_score:
        return [0] * len(scores)

    return [
        (score - min_score) / (max_score - min_score)
        for score in scores
    ]


dense_scores = normalize(dense_scores)
bm25_scores = normalize(bm25_scores)


# ==================================================
# 4. HYBRID SCORE
# ==================================================

dense_weight = 0.7
bm25_weight = 0.3

hybrid_scores = [
    dense_weight * dense +
    bm25_weight * bm25

    for dense, bm25
    in zip(dense_scores, bm25_scores)
]


# ==================================================
# 5. RESULTS
# ==================================================

results = []

for i, document in enumerate(documents):

    results.append({
        "document": document,
        "dense": dense_scores[i],
        "bm25": bm25_scores[i],
        "hybrid": hybrid_scores[i]
    })


results.sort(
    key=lambda x: x["hybrid"],
    reverse=True
)


# ==================================================
# 6. DISPLAY
# ==================================================

for rank, result in enumerate(results, start=1):

    print(f"\nRank {rank}")
    print("-" * 70)

    print("Document:", result["document"])
    print(f"Dense:   {result['dense']:.4f}")
    print(f"BM25:    {result['bm25']:.4f}")
    print(f"Hybrid:  {result['hybrid']:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Rank 1
----------------------------------------------------------------------
Document: The iPhone 15 has a powerful A16 Bionic chip and an advanced camera system.
Dense:   1.0000
BM25:    1.0000
Hybrid:  1.0000

Rank 2
----------------------------------------------------------------------
Document: The Samsung Galaxy S24 has a high resolution display and powerful processor.
Dense:   0.4445
BM25:    0.0000
Hybrid:  0.3112

Rank 3
----------------------------------------------------------------------
Document: Our store provides laptops, smartphones, tablets and other electronic products.
Dense:   0.4433
BM25:    0.0000
Hybrid:  0.3103

Rank 4
----------------------------------------------------------------------
Document: Machine learning models can be deployed using FastAPI and Docker.
Dense:   0.0000
BM25:    0.5831
Hybrid:  0.1749


## Vector Store

In [31]:
print(chunks[1].page_content)

mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.0 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature.
1 Introduction


In [32]:
print(chunks[1].metadata)

{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French t

In [43]:
from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams 
from sentence_transformers import SentenceTransformer 

# embedding model 
model = SentenceTransformer("all-MinILM-L6-v2") 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [46]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print(client.get_collections())

collections=[]


In [47]:
from qdrant_client.models import Distance, VectorParams

collection_name = "rag_documents"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

True

In [48]:
print(client.get_collection(collection_name))

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent

In [49]:
len(chunks)

51

In [50]:
# Generate embeddings of Docs 
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [53]:
from langchain_huggingface import HuggingFaceEmbeddings

embd_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [55]:
vector = embd_model.embed_query("What is artificial intelligence?")

print("Vector dimension:", len(vector))
print("First 10 values:", vector[:10])

Vector dimension: 384
First 10 values: [-0.015781084075570107, 0.014908631332218647, 0.0090955113992095, 0.03131743147969246, -0.021708263084292412, -0.04852486029267311, 0.04887938126921654, 0.028673196211457253, -0.02559715323150158, 0.054042987525463104]


In [56]:
vectors = embd_model.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Total chunks:", len(chunks))
print("Total vectors:", len(vectors))
print("Vector dimension:", len(vectors[0]))

Total chunks: 51
Total vectors: 51
Vector dimension: 384


In [57]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore(
    client=client,
    collection_name="rag_documents",
    embedding=embd_model
)

In [58]:
vector_store.add_documents(chunks)

print(f"{len(chunks)} documents added to Qdrant.")

51 documents added to Qdrant.


In [61]:
query_text = "positional embeddings"

query_embedding = embd_model.embed_query(query_text)

print("Query vector dimension:", len(query_embedding))

results = vector_store.client.query_points(
    collection_name="rag_documents",
    query=query_embedding,
    limit=3,
    with_payload=True
)

for result in results.points:
    print("\nScore:", result.score)
    print("Text:", result.payload["page_content"])

Query vector dimension: 384

Score: 0.5493469
Text: 3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5

Score: 0.42862505
Text: from layer to layer. Another way of describing this is as two convolutions with kernel size 1.
The dimensionality of input and output is dmodel = 512 , and the inner-layer has dimensionality
df f = 2048.
3.4 Embeddings and Softmax
Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embeddi

In [62]:
vector_store

In [63]:
query = "positional embeddings"

results = vector_store.similarity_search_with_score(
    query,
    k=3
)

for doc, score in results:
    print("\nScore:", score)
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)


Score: 0.5493469
Text: 3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions 

In [64]:
query = "positional embeddings"

results = vector_store.similarity_search_with_score(
    query,
    k=3
)

for doc, score in results:
    print("\nScore:", score)
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)


Score: 0.5493469
Text: 3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions 

**********************************************

# **Retrievers**

In [77]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


load_dotenv()


print("==" * 20)

loader = PyPDFLoader(r"data\pdf\attention_is_all_you_need.pdf")
docs = loader.load()

print(f"Number of docs loaded: {len(docs)}")


print("==" * 20)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print(f"Number of chunks created: {len(chunks)}")
print("Chunk size: 800")
print("Chunk overlap: 200")


print("==" * 20)

# Initialize embedding model (1536 dimensions)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding model initialized...")


# Qdrant credentials
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")


vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="retriever_demo"
)

print("Embeddings created and stored in Qdrant.")

Number of docs loaded: 11
Number of chunks created: 56
Chunk size: 800
Chunk overlap: 200
Embedding model initialized...
Embeddings created and stored in Qdrant.


In [78]:
results = vectorstore.similarity_search_with_score(
    "positional embeddings",
    k=3
)

In [85]:
results[0][0].metadata

{'producer': 'PyPDF2',
 'creator': 'PyPDF',
 'creationdate': '',
 'subject': 'Neural Information Processing Systems http://nips.cc/',
 'publisher': 'Curran Associates, Inc.',
 'language': 'en-US',
 'created': '2017',
 'eventtype': 'Poster',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-

# **Using Vector Store as Retriever** 

In [89]:
# CREATE RETRIEVER
retriever = vectorstore.as_retriever(
    search_type="similarity", 
    search_kwargs = {"k": 3}
) 

print("Retriever Created!")
query = "Positional encoding" 
results = retriever.invoke(query) 
print(f"Results: {len(results)} documents retrieved\n") 

# Show the results 
for i, doc in enumerate(results): 
    print(f"   {i+1} [{doc.metadata["page"]}]: {doc.page_content[:200]}")

Retriever Created!
Results: 3 documents retrieved

   1 [4]: order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the

   2 [4]: Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
m
   3 [5]: as the embeddings, so that the two can be summed. There are many choices of positional encodings,
learned and ﬁxed [8].
In this work, we use sine and cosine functions of different frequencies:
PE (pos


In [90]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)

In [94]:
results = retriever.invoke("positional encoding")

In [95]:
results

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

************

# Pinecone Vector store

In [1]:
from dotenv import load_dotenv 
load_dotenv() 
import os 


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") 

In [2]:
from pinecone import Pinecone 

pc = Pinecone(api_key=PINECONE_API_KEY) 

In [3]:
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [4]:
import os
from pinecone import ServerlessSpec

index_name = "rag-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

index = pc.Index(index_name)

In [5]:
index 

In [6]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [8]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)

['0c794cc0-9ce6-44cd-beb1-9e38854cbccb',
 '7bf1177e-3042-4be1-92ac-21e4d5e1cbe4',
 'b4eb0a0b-bc05-4af3-b548-00c1380ebfde',
 '1c6fce3d-2fe9-45fa-9b1e-4f879285444b',
 '7930b983-abda-4ddc-8e94-166d340f5174',
 'c0e5c0b3-6f9f-4469-8e0e-8a5056cdf6ae',
 'aac58832-c33a-43ae-9bc0-dc10011a7d2d',
 '0ab26204-a24b-4df7-9625-746bfed951b9',
 '53fe7bff-dbe0-4e02-afc4-1ff61e22d07c',
 '0ff4719a-15fc-4ca4-b6c7-7114386cf27b']

In [10]:
vector_store.delete(ids=[uuids[-1]]) # Delete the last document from pinecone 

In [11]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [12]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.570035] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [13]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='1c6fce3d-2fe9-45fa-9b1e-4f879285444b', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

# **LangFuse**

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
langfuse_handler = CallbackHandler()

print("Langfuse OK")

Langfuse OK


In [3]:
langfuse.auth_check()

True

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in one paragraph."
)

chain = prompt | llm

In [5]:
response = chain.invoke(
    {"topic": "Retrieval Augmented Generation"},
    config={
        "callbacks": [langfuse_handler]
    }
)

print(response.content)

Retrieval Augmented Generation (RAG) is a machine learning approach that combines the strengths of information retrieval and generative models to enhance the quality of natural language generation tasks. In RAG, a model first retrieves relevant documents or pieces of information from a large corpus based on a given input prompt. This retrieved information is then used to inform or guide the generative process, allowing the model to produce more accurate, contextually relevant, and factually grounded responses. By leveraging external knowledge sources in this manner, RAG enhances the capabilities of language models, making them more effective in applications such as question answering, summarization, and conversational agents.


*********************

*******************

# **Reranking in RAG Pipeline**

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

In [15]:
from pathlib import Path
pdf_path = Path("data/pdf/attention_is_all_you_need.pdf")

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Pages loaded: {len(documents)}")
print(documents[4].metadata)
print("==" * 20)
print(documents[4].page_content[:1000])

Pages loaded: 11
{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On En

In [ ]:
# Create chunks 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 40


In [18]:
print(chunks[0].page_content[:200])
print(chunks[0].metadata)

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz
{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly les

In [19]:
# Initialize embedding model 
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [23]:
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api = os.getenv("QDRANT_API_KEY") 

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=qdrant_url,
    api_key=qdrant_api,
    collection_name="attention_is_all_you_need",
)

In [24]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)

In [26]:
query = "What is the purpose of multi-head attention?"
retrieved_docs = retriever.invoke(query)

print(f"Retrieved documents: {len(retrieved_docs)}")

Retrieved documents: 10


In [27]:
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"RANK {i}")
    print(f"Page: {doc.metadata.get('page')}")
    print(doc.page_content[:1000])

RANK 1
Page: 4
MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[31, 2, 8].
RANK 2
Page: 3
Scaled Dot-Produc

In [31]:
# adding flashrank 
reranker = FlashrankRerank(
    top_n=3
)

ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [01:23<00:00, 1.24MiB/s]


In [32]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=retriever,
)

In [33]:
reranked_docs = compression_retriever.invoke(query)

print(f"Documents after reranking: {len(reranked_docs)}")

Documents after reranking: 3


In [34]:
for i, doc in enumerate(reranked_docs, start=1):
    print("=" * 80)
    print(f"RERANK {i}")
    print(f"Page: {doc.metadata.get('page')}")
    print(doc.page_content[:1000])

RERANK 1
Page: 3
a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is
much faster and more space-efﬁcient in practice, since it can be implemented using highly optimized
matrix multiplication code.
While for small values ofdk the two mechanisms perform similarly, additive attention outperforms
dot product attention without scaling for larger values ofdk [3]. We suspect that for large values of
dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has
extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk
.
3.2.2 Multi-Head Attention
Instead of performing a single attention function withdmodel-dimensional keys, values and queries,
we found it beneﬁcial to linearly project the queries, keys and valuesh times with different, learned
linear projections todk,dk anddv dimensions, respectively. On each of these projected versions of
RERANK 2
Page: 6
convolution is equa

In [35]:
def display_results(title, docs):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for i, doc in enumerate(docs, start=1):
        print(f"\nRank {i}")
        print(f"Page: {doc.metadata.get('page')}")
        print(f"Characters: {len(doc.page_content)}")
        print("-" * 80)
        print(doc.page_content[:500])

In [36]:
display_results(
    "BEFORE RERANKING - QDRANT",
    retrieved_docs
)

display_results(
    "AFTER RERANKING - FLASHRANK",
    reranked_docs
)


BEFORE RERANKING - QDRANT

Rank 1
Page: 4
Characters: 952
--------------------------------------------------------------------------------
MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of At

Rank 2
Page: 3
Characters: 957
--------------------------------------------------------------------------------
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
query with all keys, divide each by√dk, and apply a softmax f

In [37]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [38]:
context = "\n\n".join(
    doc.page_content
    for doc in reranked_docs
)

In [41]:
prompt = f"""
You are answering questions about the research paper
"Attention Is All You Need".

Answer the question using ONLY the provided context.

If the answer is not present in the context, say:
"I could not find the answer in the provided paper context."

Context:
{context}

Question:
{query}

Answer:
"""

In [42]:
response = llm.invoke(prompt)

print(response.content)

I could not find the answer in the provided paper context.


# **Hybrid RAG**

In [43]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pathlib import Path 

pdf_path = Path("data/pdf/attention_is_all_you_need.pdf")

loader = PyPDFLoader(pdf_path) 
documents = loader.load() 

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100) 

chunks = splitter.split_documents(documents) 

print(f"Loaded {len(documents)} pages") 
print(f"Created {len(chunks)} chunks") 
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

Loaded 11 pages
Created 51 chunks
Average chunk size: 690 characters


In [46]:
from dotenv import load_dotenv 
import os 
load_dotenv() 

QDRANT_URL = os.getenv("QDRANT_URL") 
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

In [49]:
from langchain_openai import OpenAIEmbeddings 
from langchain_qdrant import QdrantVectorStore 

print("Creating dense-only vector store") 

# Create embedding model 
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

# Create dense-only store 
dense_store = QdrantVectorStore.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    url=QDRANT_URL, 
    api_key=QDRANT_API_KEY, 
    collection_name="dense_only", 
    force_recreate=True
)


print("Vector store Created") 
print(f"Document indexed: {len(chunks)}")

Creating dense-only vector store
Vector store Created
Document indexed: 51


In [50]:
query = "positional encoding" 

results = dense_store.similarity_search(query, k=3) 

for i, doc in enumerate(results, 1): 
    has_match = query.lower() in doc.page_content.lower()
    print(f" {doc.page_content[:150]}...\n") 

 3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we...

 learned and ﬁxed [8].
In this work, we use sine and cosine functions of different frequencies:
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos...

 respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The...



In [52]:
# Create hybrid vector store 
from qdrant_client import QdrantClient, models 
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams 

print("Creating Hybrid vector store (dense + sparse)") 

# Initialize qdrant client 
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


# Delete collection if exists
try: 
    client.delete_collection("hybrid") 
except: 
    pass 

client.create_collection(
    collection_name="hybrid", 
    vectors_config={
        'dense': VectorParams(
            size=1536,
            distance=Distance.COSINE,
        )
    }, 
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False, 
            )
        )
    }
)


print("Hybrid Collection created")
print("Dense vectors: 1536 dimensions (cosine similarity)") 
print("Sparse vectors: BM25-style keyword matching")



# Function to create sparse vectors (Using BM-25) 
def create_sparse_vector(text: str): 
    words = text.lower().split()
    word_counts = {} 
    for word in words: 
        word = word.strip('.,!?;:"\'')
        if len(word) > 2: # Ignore every whort words 
            word_counts[word] = word_counts.get(word, 0) + 1 


    # Convert to sparse vector format with UNIQUE indices 
    # USE dict to handles hash collisions (sum values if indices collide) 
    index_values = {} 
    for word, count in word_counts.items(): 
        idx = hash(word) % 100000 
        index_values[idx] = index_values.get(idx, 0.0) + float(count) 

    # Convert to sorted lists (qdrant requires unique indices) 
    indices = sorted(index_values.keys()) 
    values = [index_values[idx] for idx in indices]

    return models.SparseVector(indices=indices, values=values) 

# Index documents with both dense and sparse vectors 
points = [] 
for i, chunk in enumerate(chunks): 
    dense_vector = embeddings.embed_query(chunk.page_content)

    # Create sparse vector 
    sparse_vector = create_sparse_vector(chunk.page_content)

    # Create point with both vectors 
    point = models.PointStruct(
        id=i, 
        vector={
            "dense": dense_vector, 
            "sparse": sparse_vector, 
        }, 
        payload={
            "text": chunk.page_content, 
            "source": chunk.metadata.get("source",""), 
        }, 
    )

    points.append(point) 

# Upload points 
client.upsert(collection_name="hybrid", points=points) 

print(f"Indexed {len(points)} documents with dense + sparse vectors")

Creating Hybrid vector store (dense + sparse)
Hybrid Collection created
Dense vectors: 384 dimensions (cosine similarity)
Sparse vectors: BM25-style keyword matching
Indexed 51 documents with dense + sparse vectors


In [54]:
# Let's test it 
query = "positional encoding" 

query_dense = embeddings.embed_query(query) 
query_sparse = create_sparse_vector(query) 

print(f"Query: {query}") 
print(f"Expected: Hybrid search should find the exact match\n") 

results_dense = client.query_points(
    collection_name="hybrid", 
    query=query_dense, 
    using="dense", 
    limit=3
)

print("Dense search results") 
for i, hit in enumerate(results_dense.points): 
    text = hit.payload["text"]
    has_match = query.lower() in text.lower() 
    match_label = "Exact math" if has_match else "No exact match"
    print(f" {i+1}, {match_label}: {text[:150]}...") 

# Search using sparse vectors 
results_sparse = client.query_points(
    collection_name="hybrid", 
    query=query_sparse,
    using="sparse", 
    limit=3
)


print("\nSparse search results.") 
for i, hit in enumerate(results_sparse.points): 
    text = hit.payload["text"] 
    has_match = query.lower() in text.lower() 
    match_label = "Exact math" if has_match else "No exact matcht"
    print(f" {i+1}. {match_label}: {text[:150]}...")


Query: positional encoding
Expected: Hybrid search should find the exact match

Dense search results
 1, Exact math: 3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we...
 2, Exact math: learned and ﬁxed [8].
In this work, we use sine and cosine functions of different frequencies:
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos...
 3, No exact match: respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The...

Sparse search results.
 1. Exact math: 1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding instead of sinusoids 4.92 25....
 2. Exact math: learned and ﬁxed [8].
In this work, we use sine and cosine functions of different frequencies:
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos...
 3. Exact math: 3.5 P

In [55]:

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever  # CORRECT: from langchain, not langchain_community

print("Building LangChain hybrid search system...\n")


bm25_retriever = BM25Retriever.from_documents(
    documents=chunks
)
bm25_retriever.k = 3

print("1. BM25Retriever created")


qdrant_retriever = dense_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("2. Qdrant Retriever created")


hybrid_retriever = EnsembleRetriever(
    retrievers=[qdrant_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # Equal weight: 50% dense, 50% sparse
)

print("3. EnsembleRetriever created (HYBRID)")
print("\nHybrid retriever ready for testing!")

Building LangChain hybrid search system...

1. BM25Retriever created
2. Qdrant Retriever created
3. EnsembleRetriever created (HYBRID)

Hybrid retriever ready for testing!


In [56]:

print("=" * 70)
print("TEST 1: Positional encoding")
print("=" * 70)
query1 = "positional encoding"
print(f"Query: '{query1}'")
print("Expected: Sparse (BM25) should excel at exact matching\n")

# Invoke all three retrievers
dense_results = qdrant_retriever.invoke(query1)
sparse_results = bm25_retriever.invoke(query1)
hybrid_results = hybrid_retriever.invoke(query1)

print("Dense (Qdrant) - Top result:")
has_match = query1.lower() in dense_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {dense_results[0].page_content[:120]}...\n")

print("Sparse (BM25) - Top result:")
has_match = query1.lower() in sparse_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {sparse_results[0].page_content[:120]}...\n")

print("Hybrid (Ensemble) - Top result:")
has_match = query1.lower() in hybrid_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {hybrid_results[0].page_content[:120]}...\n")

print("=" * 70)
print("TEST 2: decoder")
print("=" * 70)
query2 = "what docoder do?"
print(f"Query: '{query2}'")
print("Expected: Dense should excel at understanding meaning\n")

dense_results2 = qdrant_retriever.invoke(query2)
sparse_results2 = bm25_retriever.invoke(query2)
hybrid_results2 = hybrid_retriever.invoke(query2)

print("Dense (Qdrant) - Top result:")
print(f"  {dense_results2[0].page_content[:120]}...\n")

print("Sparse (BM25) - Top result:")
print(f"  {sparse_results2[0].page_content[:120]}...\n")

print("Hybrid (Ensemble) - Top result:")
print(f"  {hybrid_results2[0].page_content[:120]}...\n")

print("=" * 70)
print("CONCLUSION: Why Use EnsembleRetriever?")
print("=" * 70)
print("The Hybrid retriever (EnsembleRetriever) combines strengths:")
print("  1. Finds exact matches (like BM25)")
print("  2. Understands meaning (like dense search)")
print("  3. Works with ANY LangChain retriever")
print("  4. Production-ready with automatic RRF fusion")
print("\nThis is the recommended approach for production RAG systems!")

TEST 1: Positional encoding
Query: 'positional encoding'
Expected: Sparse (BM25) should excel at exact matching

Dense (Qdrant) - Top result:
  [EXACT MATCH]
  3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of...

Sparse (BM25) - Top result:
  [EXACT MATCH]
  1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding...

Hybrid (Ensemble) - Top result:
  [EXACT MATCH]
  learned and ﬁxed [8].
In this work, we use sine and cosine functions of different frequencies:
PE (pos,2i) =sin(pos/1000...

TEST 2: decoder
Query: 'what docoder do?'
Expected: Dense should excel at understanding meaning

Dense (Qdrant) - Top result:
  respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each...

Sparse (BM25) - Top result:
  arXiv:1609.08144, 2016.
[32] Jie Zhou, Ying Cao, Xuguang Wang, Peng Li, and Wei Xu. De